## Base models and first attempts

In [ ]:
def smiles_to_graph(smiles: str, y_val: float) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),
            atom.GetFormalCharge(),
            int(atom.GetChiralTag()),
            int(atom.GetIsAromatic())
        ]
        node_features.append(features)
    
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges.append((i, j))
        edges.append((j, i))
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    y = torch.tensor([[y_val]], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

class GraphQSARDataset(Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        df = pl.read_parquet(parquet_path).select(["canonical_smiles", "pIC50"])
        
        self.data_list = []
        for row in df.iter_rows(named=True):
            data = smiles_to_graph(row["canonical_smiles"], row["pIC50"])
            if data is not None:
                self.data_list.append(data)

    def __len__(self) -> int:
        return len(self.data_list)

    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

In [ ]:
train_graph_dataset = GraphQSARDataset(LATEST_DATA_PATH / "scaffold_train.parquet")
val_graph_dataset = GraphQSARDataset(LATEST_DATA_PATH / "scaffold_val.parquet")
train_graph_loader = DataLoader(train_graph_dataset, batch_size=128, shuffle=True)
val_graph_loader = DataLoader(val_graph_dataset, batch_size=128, shuffle=False) 

In [ ]:
class GCNBaseline(torch.nn.Module):
    def __init__(self, node_features: int = 4):
        super(GCNBaseline, self).__init__()
        
        self.conv1 = GCNConv(node_features, 64)
        self.conv2 = GCNConv(64, 64)
        self.conv3 = GCNConv(64, 64)
        
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        
        x = global_mean_pool(x, batch)
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

def evaluate_gnn(model: torch.nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return mse, mae, r2

def train_gnn(train_loader: DataLoader, val_loader: DataLoader, epochs: int = 30, lr: float = 1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GCNBaseline(node_features=4).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()

    val_mse, val_mae, val_r2 = evaluate_gnn(model, val_loader, device)
    return model, val_mse, val_mae, val_r2

In [ ]:
train_scaffold_ds = GraphQSARDataset(LATEST_DATA_PATH / "scaffold_train.parquet")
val_scaffold_ds = GraphQSARDataset(LATEST_DATA_PATH / "scaffold_val.parquet")

train_random_ds = GraphQSARDataset(LATEST_DATA_PATH / "random_train.parquet")
val_random_ds = GraphQSARDataset(LATEST_DATA_PATH / "random_val.parquet")

train_scaffold_loader = PyGDataLoader(train_scaffold_ds, batch_size=128, shuffle=True)
val_scaffold_loader = PyGDataLoader(val_scaffold_ds, batch_size=128, shuffle=False)

train_random_loader = PyGDataLoader(train_random_ds, batch_size=128, shuffle=True)
val_random_loader = PyGDataLoader(val_random_ds, batch_size=128, shuffle=False)

print("Training GNN on Scaffold Split...")
gnn_scaffold, mse_scaff, mae_scaff, r2_scaff = train_gnn(train_scaffold_loader, val_scaffold_loader)
print(f"Scaffold - MSE: {mse_scaff:.4f}, MAE: {mae_scaff:.4f}, R2: {r2_scaff:.4f}")

print("Training GNN on Random Split...")
gnn_random, mse_rand, mae_rand, r2_rand = train_gnn(train_random_loader, val_random_loader)
print(f"Random - MSE: {mse_rand:.4f}, MAE: {mae_rand:.4f}, R2: {r2_rand:.4f}")

In [ ]:
def get_mlp_metrics(model: torch.nn.Module, loader) -> tuple:
    model.eval()
    device = torch.device("cpu")
    model = model.to(device)
    
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch)
            y_true.extend(y_batch.numpy().flatten())
            y_pred.extend(preds.numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return mse, mae, r2

results = [
    {"Model": "Naive Baseline (Mean)", "Split Strategy": "Scaffold", "MSE": np.nan, "MAE": np.nan, "R2": np.nan},
    {"Model": "Naive Baseline (Mean)", "Split Strategy": "Random", "MSE": np.nan, "MAE": np.nan, "R2": np.nan},
    {"Model": "MLP Baseline (ECFP)", "Split Strategy": "Scaffold", "MSE": np.nan, "MAE": np.nan, "R2": np.nan},
    {"Model": "MLP Baseline (ECFP)", "Split Strategy": "Random", "MSE": np.nan, "MAE": np.nan, "R2": np.nan},
    {"Model": "GNN Baseline (GCN)", "Split Strategy": "Scaffold", "MSE": np.nan, "MAE": np.nan, "R2": np.nan},
    {"Model": "GNN Baseline (GCN)", "Split Strategy": "Random", "MSE": np.nan, "MAE": np.nan, "R2": np.nan}
]

mlp_scaffold_mse, mlp_scaffold_mae, mlp_scaffold_r2 = get_mlp_metrics(trained_model, val_loader)

train_random_ecfp = QsARDataset(LATEST_DATA_PATH / "random_train.parquet")
val_random_ecfp = QsARDataset(LATEST_DATA_PATH / "random_val.parquet")

train_loader_random = DataLoader(train_random_ecfp, batch_size=128, shuffle=True)
val_loader_random = DataLoader(val_random_ecfp, batch_size=128, shuffle=False)

print("Training MLP on Random Split...")
trained_model_random = train_model(train_loader_random, val_loader_random, epochs=30, lr=1e-3)

mlp_random_mse, mlp_random_mae, mlp_random_r2 = get_mlp_metrics(trained_model_random, val_loader_random)

results[2]["MSE"] = mlp_scaffold_mse
results[2]["MAE"] = mlp_scaffold_mae
results[2]["R2"] = mlp_scaffold_r2

results[3]["MSE"] = mlp_random_mse
results[3]["MAE"] = mlp_random_mae
results[3]["R2"] = mlp_random_r2

final_df = pd.DataFrame(results).round(4)
print("\n--- FINAL BASELINE REPORT ---")
print(final_df)

final_df.to_csv("baseline_metrics_report.csv", index=False)

In [ ]:
def smiles_to_graph_advanced(smiles: str, y_val: float) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            atom.GetMass()
        ]
        node_features.append(features)
    
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        
        bond_type = bond.GetBondTypeAsDouble()
        
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([[bond_type], [bond_type]])
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    y = torch.tensor([[y_val]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

class GATBaseline(torch.nn.Module):
    def __init__(self, node_features: int = 6, edge_features: int = 1):
        super(GATBaseline, self).__init__()
        
        self.conv1 = GATConv(node_features, 64, edge_dim=edge_features)
        self.conv2 = GATConv(64, 64, edge_dim=edge_features)
        self.conv3 = GATConv(64, 64, edge_dim=edge_features)
        
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = F.elu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = F.elu(self.conv2(x, edge_index, edge_attr=edge_attr))
        x = F.elu(self.conv3(x, edge_index, edge_attr=edge_attr))
        
        x = global_mean_pool(x, batch)
        
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
def smiles_to_graph_advanced(smiles: str, y_val: float) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            atom.GetMass()
        ]
        node_features.append(features)
    
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondTypeAsDouble()
        
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([[bond_type], [bond_type]])
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    y = torch.tensor([[y_val]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

class GraphQSARDatasetAdvanced(Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        df = pl.read_parquet(parquet_path).select(["canonical_smiles", "pIC50"])
        
        self.data_list = []
        for row in df.iter_rows(named=True):
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"])
            if data is not None:
                self.data_list.append(data)

    def __len__(self) -> int:
        return len(self.data_list)

    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

class GATBaseline(torch.nn.Module):
    def __init__(self, node_features: int = 6, edge_features: int = 1):
        super(GATBaseline, self).__init__()
        
        self.conv1 = GATConv(node_features, 64, edge_dim=edge_features)
        self.conv2 = GATConv(64, 64, edge_dim=edge_features)
        self.conv3 = GATConv(64, 64, edge_dim=edge_features)
        
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = F.elu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = F.elu(self.conv2(x, edge_index, edge_attr=edge_attr))
        x = F.elu(self.conv3(x, edge_index, edge_attr=edge_attr))
        
        x = global_mean_pool(x, batch)
        
        x = F.relu(self.fc1(x))
        return self.fc2(x)

def evaluate_gat(model: torch.nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return mse, mae, r2

def train_gat(train_loader: DataLoader, val_loader: DataLoader, epochs: int = 30, lr: float = 1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GATBaseline(node_features=6, edge_features=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * data.num_graphs

        avg_train = train_loss / len(train_loader.dataset)
        
        if epoch % 5 == 0 or epoch == epochs - 1:
            val_mse, val_mae, val_r2 = evaluate_gat(model, val_loader, device)
            print(f"Epoch {epoch:02d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f}")

    val_mse, val_mae, val_r2 = evaluate_gat(model, val_loader, device)
    return model, val_mse, val_mae, val_r2

train_adv_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
val_adv_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")


train_adv_loader = PyGDataLoader(train_adv_ds, batch_size=128, shuffle=True)
val_adv_loader = PyGDataLoader(val_adv_ds, batch_size=128, shuffle=False)

print("Training GAT on Scaffold Split...")
gat_model, gat_mse, gat_mae, gat_r2 = train_gat(train_adv_loader, val_adv_loader)
print(f"Final GAT Scaffold - MSE: {gat_mse:.4f}, MAE: {gat_mae:.4f}, R2: {gat_r2:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, global_mean_pool
from rdkit import Chem
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path
from typing import Union
import os

def smiles_to_graph_advanced(smiles: str, y_val: float) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            atom.GetMass()
        ]
        node_features.append(features)
    
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondTypeAsDouble()
        
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([[bond_type], [bond_type]])
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    y = torch.tensor([[y_val]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

class GraphQSARDatasetAdvanced(Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        df = pl.read_parquet(parquet_path).select(["canonical_smiles", "pIC50"])
        
        self.data_list = []
        for row in df.iter_rows(named=True):
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"])
            if data is not None:
                self.data_list.append(data)

    def __len__(self) -> int:
        return len(self.data_list)

    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

class GINBaseline(torch.nn.Module):
    def __init__(self, node_features: int = 6, edge_features: int = 1, hidden_dim: int = 64):
        super(GINBaseline, self).__init__()
        
        self.node_emb = nn.Linear(node_features, hidden_dim)
        self.edge_emb = nn.Linear(edge_features, hidden_dim)
        
        self.conv1 = GINEConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU()
            )
        )
        
        self.conv2 = GINEConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU()
            )
        )
        
        self.conv3 = GINEConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU()
            )
        )
        
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        
        x = self.conv1(x, edge_index, edge_attr)
        x = self.conv2(x, edge_index, edge_attr)
        x = self.conv3(x, edge_index, edge_attr)
        
        x = global_mean_pool(x, batch)
        
        x = F.relu(self.fc1(x))
        return self.fc2(x)

def evaluate_gin(model: torch.nn.Module, loader: PyGDataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return mse, mae, r2

def train_gin(train_loader: PyGDataLoader, val_loader: PyGDataLoader, epochs: int = 30, lr: float = 1e-3, run_name: str = "GIN_Run"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GINBaseline(node_features=6, edge_features=1, hidden_dim=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()

    train_losses = []
    val_losses = []
    val_r2_scores = []

    mlflow.set_experiment("GNN_GIN_Experiments")
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": lr,
            "hidden_dim": 64,
            "model_architecture": "GIN",
            "split_type": run_name
        })

        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * data.num_graphs

            avg_train = train_loss / len(train_loader.dataset)
            val_mse, val_mae, val_r2 = evaluate_gin(model, val_loader, device)

            train_losses.append(avg_train)
            val_losses.append(val_mse)
            val_r2_scores.append(val_r2)

            mlflow.log_metric("train_mse", avg_train, step=epoch)
            mlflow.log_metric("val_mse", val_mse, step=epoch)
            mlflow.log_metric("val_mae", val_mae, step=epoch)
            mlflow.log_metric("val_r2", val_r2, step=epoch)
            
            if epoch % 5 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:02d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f}")

        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train MSE', color='#4C72B0', linewidth=2)
        plt.plot(val_losses, label='Val MSE', color='#DD8452', linewidth=2)
        plt.title(f'Learning Curve: {run_name}', fontsize=14)
        plt.xlabel('Epochs', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        
        plot_filename = f"learning_curve_{run_name.replace(' ', '_')}.png"
        plt.savefig(plot_filename)
        plt.close()

        mlflow.log_artifact(plot_filename)
        
        mlflow.pytorch.log_model(model, "model")
        
        if os.path.exists(plot_filename):
            os.remove(plot_filename)

    return model, val_losses[-1], val_mae, val_r2_scores[-1]


# LATEST_DATA_PATH = get_latest_data_version(GOLD_DIR)

train_adv_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
val_adv_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")

train_adv_loader = PyGDataLoader(train_adv_ds, batch_size=128, shuffle=True)
val_adv_loader = PyGDataLoader(val_adv_ds, batch_size=128, shuffle=False)

print("Training GIN on Scaffold Split...")
gin_model, gin_mse, gin_mae, gin_r2 = train_gin(train_adv_loader, val_adv_loader, run_name="Scaffold_Split")
print(f"Final GIN Scaffold - MSE: {gin_mse:.4f}, MAE: {gin_mae:.4f}, R2: {gin_r2:.4f}\n")

train_adv_random_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "random_train.parquet")
val_adv_random_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "random_val.parquet")

train_adv_random_loader = PyGDataLoader(train_adv_random_ds, batch_size=128, shuffle=True)
val_adv_random_loader = PyGDataLoader(val_adv_random_ds, batch_size=128, shuffle=False)

print("Training GIN on Random Split...")
gin_model_random, gin_mse_rand, gin_mae_rand, gin_r2_rand = train_gin(train_adv_random_loader, val_adv_random_loader, run_name="Random_Split")
print(f"Final GIN Random - MSE: {gin_mse_rand:.4f}, MAE: {gin_mae_rand:.4f}, R2: {gin_r2_rand:.4f}")

In [ ]:
import os
from pathlib import Path
from typing import Union
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_mean_pool
from rdkit import Chem
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def one_hot_encoding(value, choices: list) -> list:
    encoding = [0] * (len(choices) + 1)
    index = choices.index(value) if value in choices else -1
    encoding[index] = 1
    return encoding

def smiles_to_graph_advanced(smiles: str, y_val: float) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    Chem.SanitizeMol(mol)
    node_features = []
    for atom in mol.GetAtoms():
        features = (
            one_hot_encoding(atom.GetAtomicNum(), [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]) +
            one_hot_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5]) +
            one_hot_encoding(atom.GetFormalCharge(), [-1, 0, 1]) +
            one_hot_encoding(int(atom.GetHybridization()), [2, 3, 4]) +
            [1 if atom.GetIsAromatic() else 0] +
            [atom.GetMass() / 100.0]
        )
        node_features.append(features)
    x = torch.tensor(node_features, dtype=torch.float)
    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondType()
        b_features = one_hot_encoding(
            bond_type,
            [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
        )
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([b_features, b_features])
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 5), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
    y = torch.tensor([[y_val]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

class GraphQSARDatasetAdvanced(torch.utils.data.Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        df = pl.read_parquet(parquet_path).select(["canonical_smiles", "pIC50"])
        self.data_list = []
        for row in df.iter_rows(named=True):
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"])
            if data is not None:
                self.data_list.append(data)
    def __len__(self) -> int:
        return len(self.data_list)
    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

class GINOptimized(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.2):
        super(GINOptimized, self).__init__()
        self.dropout = dropout
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(num_layers):
            nn_seq = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.BatchNorm1d(hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim)
            )
            self.convs.append(GINEConv(nn_seq))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        for conv, bn in zip(self.convs, self.batch_norms):
            x_res = x
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res
        x = global_mean_pool(x, batch)
        return self.mlp(x)

def evaluate_gin(model: nn.Module, loader: PyGDataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, mae, r2

def train_gin(train_loader: PyGDataLoader, val_loader: PyGDataLoader, node_dim: int, edge_dim: int, epochs: int = 30, lr: float = 1e-3, run_name: str = "GIN_Run"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GINOptimized(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=128, num_layers=4, dropout=0.2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    train_losses = []
    val_losses = []
    val_r2_scores = []
    mlflow.set_experiment("GNN_GIN_Experiments")
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": lr,
            "hidden_dim": 128,
            "model_architecture": "GIN_Optimized",
            "split_type": run_name
        })
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * data.num_graphs
            avg_train = train_loss / len(train_loader.dataset)
            val_mse, val_mae, val_r2 = evaluate_gin(model, val_loader, device)
            train_losses.append(avg_train)
            val_losses.append(val_mse)
            val_r2_scores.append(val_r2)
            mlflow.log_metric("train_mse", avg_train, step=epoch)
            mlflow.log_metric("val_mse", val_mse, step=epoch)
            mlflow.log_metric("val_mae", val_mae, step=epoch)
            mlflow.log_metric("val_r2", val_r2, step=epoch)
            if epoch % 5 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:02d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f}")
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train MSE', color='#4C72B0', linewidth=2)
        plt.plot(val_losses, label='Val MSE', color='#DD8452', linewidth=2)
        plt.title(f'Learning Curve: {run_name}', fontsize=14)
        plt.xlabel('Epochs', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plot_filename = f"learning_curve_{run_name.replace(' ', '_')}.png"
        plt.savefig(plot_filename)
        plt.close()
        mlflow.log_artifact(plot_filename)
        mlflow.pytorch.log_model(model, "model")
        if os.path.exists(plot_filename):
            os.remove(plot_filename)
    return model, val_losses[-1], val_mae, val_r2_scores[-1]

In [ ]:
if __name__ == "__main__":
    LATEST_DATA_PATH = get_latest_valid_version(GOLD_DIR)
    
    train_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
    val_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")
    
    train_random_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "random_train.parquet")
    val_random_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "random_val.parquet")
    
    train_scaffold_loader = PyGDataLoader(train_scaffold_ds, batch_size=128, shuffle=True)
    val_scaffold_loader = PyGDataLoader(val_scaffold_ds, batch_size=128, shuffle=False)
    
    train_random_loader = PyGDataLoader(train_random_ds, batch_size=128, shuffle=True)
    val_random_loader = PyGDataLoader(val_random_ds, batch_size=128, shuffle=False)
    
    sample_batch = next(iter(train_scaffold_loader))
    n_dim = sample_batch.num_node_features
    e_dim = sample_batch.num_edge_features
    
    print("Training GIN on Scaffold Split...")
    model_scaff, mse_scaff, mae_scaff, r2_scaff = train_gin(
        train_scaffold_loader, val_scaffold_loader, node_dim=n_dim, edge_dim=e_dim, run_name="Scaffold_Split"
    )
    print(f"Final GIN Scaffold - MSE: {mse_scaff:.4f}, MAE: {mae_scaff:.4f}, R2: {r2_scaff:.4f}\n")
    
    print("Training GIN on Random Split...")
    model_rand, mse_rand, mae_rand, r2_rand = train_gin(
        train_random_loader, val_random_loader, node_dim=n_dim, edge_dim=e_dim, run_name="Random_Split"
    )
    print(f"Final GIN Random - MSE: {mse_rand:.4f}, MAE: {mae_rand:.4f}, R2: {r2_rand:.4f}")

In [ ]:
import os
from pathlib import Path
from typing import Union
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_max_pool, global_add_pool
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Inicjalizacja narzędzia do usuwania soli (np. chlorków, sodu) z cząsteczek
remover = SaltRemover()

def one_hot_encoding(value, choices: list) -> list:
    encoding = [0] * (len(choices) + 1)
    index = choices.index(value) if value in choices else -1
    encoding[index] = 1
    return encoding

def smiles_to_graph_advanced(smiles: str, y_val: float) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
        
    # Salt stripping i sanityzacja
    try:
        res = remover.StripMol(mol, dontRemoveEverything=True)
        if res is not None:
            mol = res
        Chem.SanitizeMol(mol)
    except:
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = (
            one_hot_encoding(atom.GetAtomicNum(), [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]) +
            one_hot_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5]) +
            one_hot_encoding(atom.GetFormalCharge(), [-1, 0, 1]) +
            one_hot_encoding(int(atom.GetHybridization()), [2, 3, 4]) +
            [1 if atom.GetIsAromatic() else 0] +
            [atom.GetMass() / 100.0]
        )
        node_features.append(features)
        
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondType()
        b_features = one_hot_encoding(
            bond_type,
            [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
        )
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([b_features, b_features])
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 5), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
        
    y = torch.tensor([[y_val]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

class GraphQSARDatasetAdvanced(torch.utils.data.Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        df = pl.read_parquet(parquet_path).select(["canonical_smiles", "pIC50"])
        self.data_list = []
        for row in df.iter_rows(named=True):
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"])
            if data is not None:
                self.data_list.append(data)
                
    def __len__(self) -> int:
        return len(self.data_list)
        
    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

class GINAdvanced(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.2):
        super(GINAdvanced, self).__init__()
        self.dropout = dropout
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.virtual_node_emb = nn.Embedding(1, hidden_dim)
        
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.mlp_virtual_nodes = nn.ModuleList()
        
        for _ in range(num_layers):
            nn_seq = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.BatchNorm1d(hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim)
            )
            self.convs.append(GINEConv(nn_seq))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
            self.mlp_virtual_nodes.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.BatchNorm1d(hidden_dim * 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU()
                )
            )
            
        self.pool_concat_dim = hidden_dim * 3
        
        self.mlp = nn.Sequential(
            nn.Linear(self.pool_concat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        batch_size = batch.max().item() + 1
        virtual_node = self.virtual_node_emb(torch.zeros(batch_size, dtype=torch.long, device=x.device))
        
        for i, (conv, bn) in enumerate(zip(self.convs, self.batch_norms)):
            x_res = x
            virtual_node_expanded = virtual_node[batch]
            x = x + virtual_node_expanded
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res
            if i < len(self.convs) - 1:
                virtual_node_pool = global_add_pool(x, batch)
                virtual_node = virtual_node + F.dropout(
                    self.mlp_virtual_nodes[i](virtual_node_pool),
                    p=self.dropout,
                    training=self.training
                )
                
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_add = global_add_pool(x, batch)
        x_pooled = torch.cat([x_mean, x_max, x_add], dim=1)
        return self.mlp(x_pooled)

def evaluate_gin(model: nn.Module, loader: PyGDataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, mae, r2

def train_gin(train_loader: PyGDataLoader, val_loader: PyGDataLoader, node_dim: int, edge_dim: int, epochs: int = 100, lr: float = 1e-3, run_name: str = "GIN_Run"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GINAdvanced(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=128, num_layers=4, dropout=0.2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    val_r2_scores = []
    
    mlflow.set_experiment("GNN_GIN_Experiments")
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": lr,
            "hidden_dim": 128,
            "model_architecture": "GIN_Advanced_VN",
            "split_type": run_name
        })
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * data.num_graphs
                
            avg_train = train_loss / len(train_loader.dataset)
            val_mse, val_mae, val_r2 = evaluate_gin(model, val_loader, device)
            scheduler.step(val_mse)
            
            train_losses.append(avg_train)
            val_losses.append(val_mse)
            val_r2_scores.append(val_r2)
            current_lr = optimizer.param_groups[0]['lr']
            
            mlflow.log_metric("train_mse", avg_train, step=epoch)
            mlflow.log_metric("val_mse", val_mse, step=epoch)
            mlflow.log_metric("val_mae", val_mae, step=epoch)
            mlflow.log_metric("val_r2", val_r2, step=epoch)
            mlflow.log_metric("lr", current_lr, step=epoch)
            
            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:03d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f} | LR {current_lr}")
                
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train MSE', color='#4C72B0', linewidth=2)
        plt.plot(val_losses, label='Val MSE', color='#DD8452', linewidth=2)
        plt.title(f'Learning Curve: {run_name}', fontsize=14)
        plt.xlabel('Epochs', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plot_filename = f"learning_curve_{run_name.replace(' ', '_')}.png"
        plt.savefig(plot_filename)
        plt.close()
        
        mlflow.log_artifact(plot_filename)
        mlflow.pytorch.log_model(model, "model")
        if os.path.exists(plot_filename):
            os.remove(plot_filename)
            
    return model, val_losses[-1], val_mae, val_r2_scores[-1]

In [ ]:
if __name__ == "__main__":
    LATEST_DATA_PATH = get_latest_valid_version(GOLD_DIR)
    
    print("Loading datasets...")
    train_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
    val_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")
    
    train_scaffold_loader = PyGDataLoader(train_scaffold_ds, batch_size=128, shuffle=True)
    val_scaffold_loader = PyGDataLoader(val_scaffold_ds, batch_size=128, shuffle=False)
    
    # Pobranie wymiarów cech
    sample_batch = next(iter(train_scaffold_loader))
    n_dim = sample_batch.num_node_features
    e_dim = sample_batch.num_edge_features
    
    print("Training GIN Advanced on Scaffold Split...")
    model_scaff, mse_scaff, mae_scaff, r2_scaff = train_gin(
        train_scaffold_loader, val_scaffold_loader, node_dim=n_dim, edge_dim=e_dim, epochs=100, run_name="Scaffold_Split_Advanced"
    )
    print(f"\nFinal GIN Scaffold - MSE: {mse_scaff:.4f}, MAE: {mae_scaff:.4f}, R2: {r2_scaff:.4f}")

In [ ]:
import os
from pathlib import Path
from typing import Union
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_max_pool, global_add_pool
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Inicjalizacja narzędzia do usuwania soli
remover = SaltRemover()

def one_hot_encoding(value, choices: list) -> list:
    encoding = [0] * (len(choices) + 1)
    index = choices.index(value) if value in choices else -1
    encoding[index] = 1
    return encoding

def smiles_to_graph_advanced(smiles: str, y_val: float, global_features: list) -> Union[Data, None]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
        
    try:
        res = remover.StripMol(mol, dontRemoveEverything=True)
        if res is not None:
            mol = res
        Chem.SanitizeMol(mol)
    except:
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = (
            one_hot_encoding(atom.GetAtomicNum(), [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]) +
            one_hot_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5]) +
            one_hot_encoding(atom.GetFormalCharge(), [-1, 0, 1]) +
            one_hot_encoding(int(atom.GetHybridization()), [2, 3, 4]) +
            [1 if atom.GetIsAromatic() else 0] +
            [atom.GetMass() / 100.0]
        )
        node_features.append(features)
        
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondType()
        b_features = one_hot_encoding(
            bond_type,
            [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
        )
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([b_features, b_features])
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 5), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
        
    y = torch.tensor([[y_val]], dtype=torch.float)
    
    # Przygotowanie deskryptorów globalnych
    # global_features to: [aromatic_rings, hbd, alogp, mw_freebase, rtb, psa, hba, qed_weighted]
    scaled_globals = [
        global_features[0],               # aromatic_rings
        global_features[1],               # hbd
        global_features[2],               # alogp
        global_features[3] / 100.0,       # mw_freebase (skalowanie)
        global_features[4],               # rtb
        global_features[5] / 100.0,       # psa (skalowanie)
        global_features[6],               # hba
        global_features[7]                # qed_weighted
    ]
    
    global_tensor = torch.tensor([scaled_globals], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, global_feats=global_tensor)

class GraphQSARDatasetAdvanced(torch.utils.data.Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        
        cols = ["canonical_smiles", "pIC50", "aromatic_rings", "hbd", "alogp", "mw_freebase", "rtb", "psa", "hba", "qed_weighted"]
        df = pl.read_parquet(parquet_path).select(cols).drop_nulls()
        
        self.data_list = []
        for row in df.iter_rows(named=True):
            g_feats = [
                row["aromatic_rings"], row["hbd"], row["alogp"], row["mw_freebase"], 
                row["rtb"], row["psa"], row["hba"], row["qed_weighted"]
            ]
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"], g_feats)
            if data is not None:
                self.data_list.append(data)
                
    def __len__(self) -> int:
        return len(self.data_list)
        
    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

class GINAdvanced(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, global_dim: int = 8, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.2):
        super(GINAdvanced, self).__init__()
        self.dropout = dropout
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.virtual_node_emb = nn.Embedding(1, hidden_dim)
        
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.mlp_virtual_nodes = nn.ModuleList()
        
        for _ in range(num_layers):
            nn_seq = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.BatchNorm1d(hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim)
            )
            self.convs.append(GINEConv(nn_seq))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
            self.mlp_virtual_nodes.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.BatchNorm1d(hidden_dim * 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU()
                )
            )
            
        self.pool_concat_dim = (hidden_dim * 3) + global_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(self.pool_concat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor, global_feats: torch.Tensor) -> torch.Tensor:
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        batch_size = batch.max().item() + 1
        virtual_node = self.virtual_node_emb(torch.zeros(batch_size, dtype=torch.long, device=x.device))
        
        for i, (conv, bn) in enumerate(zip(self.convs, self.batch_norms)):
            x_res = x
            virtual_node_expanded = virtual_node[batch]
            x = x + virtual_node_expanded
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res
            if i < len(self.convs) - 1:
                virtual_node_pool = global_add_pool(x, batch)
                virtual_node = virtual_node + F.dropout(
                    self.mlp_virtual_nodes[i](virtual_node_pool),
                    p=self.dropout,
                    training=self.training
                )
                
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_add = global_add_pool(x, batch)
        
        # Konkatenacja z deskryptorami
        x_pooled = torch.cat([x_mean, x_max, x_add, global_feats], dim=1)
        return self.mlp(x_pooled)

def evaluate_gin(model: nn.Module, loader: PyGDataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch, data.global_feats)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, mae, r2

def train_gin(train_loader: PyGDataLoader, val_loader: PyGDataLoader, node_dim: int, edge_dim: int, global_dim: int, epochs: int = 100, lr: float = 1e-3, run_name: str = "GIN_Run"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GINAdvanced(node_dim=node_dim, edge_dim=edge_dim, global_dim=global_dim, hidden_dim=128, num_layers=4, dropout=0.2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    val_r2_scores = []
    
    mlflow.set_experiment("GNN_GIN_Experiments")
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": lr,
            "hidden_dim": 128,
            "model_architecture": "GIN_Advanced_VN_Descriptors",
            "split_type": run_name
        })
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.batch, data.global_feats)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * data.num_graphs
                
            avg_train = train_loss / len(train_loader.dataset)
            val_mse, val_mae, val_r2 = evaluate_gin(model, val_loader, device)
            scheduler.step(val_mse)
            
            train_losses.append(avg_train)
            val_losses.append(val_mse)
            val_r2_scores.append(val_r2)
            current_lr = optimizer.param_groups[0]['lr']
            
            mlflow.log_metric("train_mse", avg_train, step=epoch)
            mlflow.log_metric("val_mse", val_mse, step=epoch)
            mlflow.log_metric("val_mae", val_mae, step=epoch)
            mlflow.log_metric("val_r2", val_r2, step=epoch)
            mlflow.log_metric("lr", current_lr, step=epoch)
            
            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:03d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f} | LR {current_lr}")
                
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train MSE', color='#4C72B0', linewidth=2)
        plt.plot(val_losses, label='Val MSE', color='#DD8452', linewidth=2)
        plt.title(f'Learning Curve: {run_name}', fontsize=14)
        plt.xlabel('Epochs', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plot_filename = f"learning_curve_{run_name.replace(' ', '_')}.png"
        plt.savefig(plot_filename)
        plt.close()
        
        mlflow.log_artifact(plot_filename)
        mlflow.pytorch.log_model(model, "model")
        if os.path.exists(plot_filename):
            os.remove(plot_filename)
            
    return model, val_losses[-1], val_mae, val_r2_scores[-1]

if __name__ == "__main__":
    LATEST_DATA_PATH = get_latest_valid_version(GOLD_DIR)
    
    print("Loading datasets...")
    train_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
    val_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")
    
    train_scaffold_loader = PyGDataLoader(train_scaffold_ds, batch_size=128, shuffle=True)
    val_scaffold_loader = PyGDataLoader(val_scaffold_ds, batch_size=128, shuffle=False)
    
    # Automatyczne rozpoznanie wymiarów
    sample_batch = next(iter(train_scaffold_loader))
    n_dim = sample_batch.num_node_features
    e_dim = sample_batch.num_edge_features
    g_dim = sample_batch.global_feats.shape[1] # Powinno wynosić 8
    
    print(f"Features: Node {n_dim}, Edge {e_dim}, Global {g_dim}")
    print("Training GIN Advanced + Descriptors on Scaffold Split...")
    
    model_scaff, mse_scaff, mae_scaff, r2_scaff = train_gin(
        train_scaffold_loader, val_scaffold_loader, 
        node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim, 
        epochs=100, run_name="Scaffold_Split_Advanced_Desc"
    )
    print(f"\nFinal GIN Scaffold - MSE: {mse_scaff:.4f}, MAE: {mae_scaff:.4f}, R2: {r2_scaff:.4f}")

## Base model

In [2]:
import os
from pathlib import Path

import mlflow
import polars as pl
import torch

from typing import Union

import matplotlib.pyplot as plt
import mlflow.pytorch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_add_pool, global_max_pool, global_mean_pool

BASE_DIR = Path(r"C:\Users\User\Desktop\spark_airflow\chembl\dags\data")
GOLD_DIR = BASE_DIR / "gold"
MLFLOW_TRACKING_URI = "http://localhost:5000"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow tracking set to: {MLFLOW_TRACKING_URI}")

def get_latest_valid_version(base_path: Path, prefix: str = "version_"):
    if not base_path.exists():
        return None
    
    subdirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith(prefix)]
    subdirs.sort(key=lambda d: d.name, reverse=True)
    
    for subdir in subdirs:
        if (subdir / "scaffold_train.parquet").exists() and (subdir / "scaffold_val.parquet").exists():
            return subdir
            
    return None

LATEST_DATA_PATH = get_latest_valid_version(GOLD_DIR)

if LATEST_DATA_PATH:
    print(f"Ready for training. Data found: {LATEST_DATA_PATH.name}")
else:
    print("Data not found. Ensure the EDA notebook generated the files in the gold directory.")

MLflow tracking set to: http://localhost:5000
Ready for training. Data found: version_v2.0_20260606_1021


In [3]:
def one_hot_encoding(value, choices: list):
    encoding = [0] * (len(choices) + 1)
    index = choices.index(value) if value in choices else -1
    encoding[index] = 1
    return encoding

def smiles_to_graph_advanced(smiles: str, y_val: float, global_features: list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: 
        return None

    node_features = []
    for atom in mol.GetAtoms():
        features = (
            one_hot_encoding(atom.GetAtomicNum(), [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]) +
            one_hot_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5]) +
            one_hot_encoding(atom.GetFormalCharge(), [-1, 0, 1]) +
            one_hot_encoding(int(atom.GetHybridization()), [2, 3, 4]) +
            [1 if atom.GetIsAromatic() else 0] +
            [atom.GetMass() / 100.0]
        )
        node_features.append(features)
        
    x = torch.tensor(node_features, dtype=torch.float)

    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bond_type = bond.GetBondType()
        b_features = one_hot_encoding(
            bond_type,
            [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
        )
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([b_features, b_features])
        
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 5), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
        
    y = torch.tensor([[y_val]], dtype=torch.float)
    global_tensor = torch.tensor([global_features], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, global_feats=global_tensor)

class GraphQSARDatasetAdvanced(torch.utils.data.Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        
        cols = ["canonical_smiles", "pIC50", "aromatic_rings", "hbd", "alogp", "mw_freebase", "rtb", "psa", "hba", "qed_weighted"]
        df = pl.read_parquet(parquet_path).select(cols).drop_nulls()
        
        self.data_list = []
        for row in df.iter_rows(named=True):
            g_feats = [
                row["aromatic_rings"], row["hbd"], row["alogp"], row["mw_freebase"], 
                row["rtb"], row["psa"], row["hba"], row["qed_weighted"]
            ]
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"], g_feats)
            if data is not None:
                self.data_list.append(data)
                
    def __len__(self) -> int:
        return len(self.data_list)
        
    def __getitem__(self, idx: int) -> Data:
        return self.data_list[idx]

class GINAdvanced(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, global_dim: int = 8, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.2):
        super(GINAdvanced, self).__init__()
        self.dropout = dropout
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.virtual_node_emb = nn.Embedding(1, hidden_dim)
        
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.mlp_virtual_nodes = nn.ModuleList()
        
        for _ in range(num_layers):
            nn_seq = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.BatchNorm1d(hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim)
            )
            self.convs.append(GINEConv(nn_seq))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
            self.mlp_virtual_nodes.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.BatchNorm1d(hidden_dim * 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU()
                )
            )
            
        self.global_bn = nn.BatchNorm1d(global_dim)
        self.pool_concat_dim = (hidden_dim * 3) + global_dim
        
        self.mlp = nn.Sequential(
            nn.Linear(self.pool_concat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor, batch: torch.Tensor, global_feats: torch.Tensor) -> torch.Tensor:
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        batch_size = batch.max().item() + 1
        virtual_node = self.virtual_node_emb(torch.zeros(batch_size, dtype=torch.long, device=x.device))
        
        for i, (conv, bn) in enumerate(zip(self.convs, self.batch_norms)):
            x_res = x
            virtual_node_expanded = virtual_node[batch]
            x = x + virtual_node_expanded
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res
            if i < len(self.convs) - 1:
                virtual_node_pool = global_add_pool(x, batch)
                virtual_node = virtual_node + F.dropout(
                    self.mlp_virtual_nodes[i](virtual_node_pool),
                    p=self.dropout,
                    training=self.training
                )
                
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_add = global_add_pool(x, batch)
        
        global_feats_norm = self.global_bn(global_feats)
        x_pooled = torch.cat([x_mean, x_max, x_add, global_feats_norm], dim=1)
        return self.mlp(x_pooled)

def evaluate_gin(model: nn.Module, loader: PyGDataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch, data.global_feats)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, mae, r2

def train_gin(train_loader: PyGDataLoader, val_loader: PyGDataLoader, node_dim: int, edge_dim: int, global_dim: int, epochs: int = 100, lr: float = 1e-3, run_name: str = "GIN_Run"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Starting computation on: {device}")
    
    model = GINAdvanced(node_dim=node_dim, edge_dim=edge_dim, global_dim=global_dim, hidden_dim=128, num_layers=4, dropout=0.2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    
    mlflow.set_experiment("GNN_GIN_Experiments")
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": lr,
            "hidden_dim": 128,
            "model_architecture": "GIN_Advanced_Clean_Desc",
        })
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.batch, data.global_feats)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * data.num_graphs
                
            avg_train = train_loss / len(train_loader.dataset)
            val_mse, val_mae, val_r2 = evaluate_gin(model, val_loader, device)
            scheduler.step(val_mse)
            
            train_losses.append(avg_train)
            val_losses.append(val_mse)
            
            mlflow.log_metric("train_mse", avg_train, step=epoch)
            mlflow.log_metric("val_mse", val_mse, step=epoch)
            mlflow.log_metric("val_mae", val_mae, step=epoch)
            mlflow.log_metric("val_r2", val_r2, step=epoch)
            
            if epoch % 5 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:03d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f}")
                
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train MSE', color='#4C72B0', linewidth=2)
        plt.plot(val_losses, label='Val MSE', color='#DD8452', linewidth=2)
        plt.title(f'Learning Curve: {run_name}', fontsize=14)
        plt.xlabel('Epochs', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plot_filename = f"learning_curve_{run_name.replace(' ', '_')}.png"
        plt.savefig(plot_filename)
        plt.close()
        
        mlflow.log_artifact(plot_filename)
        mlflow.pytorch.log_model(model, "model")
        
        if os.path.exists(plot_filename):
            os.remove(plot_filename)
            
    return model, val_losses[-1], val_mae, val_r2

In [4]:
print(f"Loading data from: {LATEST_DATA_PATH.name}...")

train_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
val_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")
    
train_scaffold_loader = PyGDataLoader(
    train_scaffold_ds, batch_size=128, shuffle=True, num_workers=0, pin_memory=False
)
val_scaffold_loader = PyGDataLoader(
    val_scaffold_ds, batch_size=128, shuffle=False, num_workers=0, pin_memory=False
)
    
sample_batch = next(iter(train_scaffold_loader))
n_dim = sample_batch.num_node_features
e_dim = sample_batch.num_edge_features
g_dim = sample_batch.global_feats.shape[1]
    
print(f"Detected features -> Node: {n_dim}, Edge: {e_dim}, Global: {g_dim}")
print("Starting GIN Advanced training...")
    
model_scaff, mse_scaff, mae_scaff, r2_scaff = train_gin(
    train_scaffold_loader, val_scaffold_loader, 
    node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim, 
    epochs=60, run_name="Scaffold_Split_Clean_Desc"
)

print(f"\nFINAL RESULT - MSE: {mse_scaff:.4f}, MAE: {mae_scaff:.4f}, R2: {r2_scaff:.4f}")

Loading data from: version_v2.0_20260606_1021...
Detected features -> Node: 28, Edge: 5, Global: 8
Starting GIN Advanced training...
Starting computation on: cpu
Epoch 000: Train MSE 34.6165 | Val MSE 19.0313 | Val R2 -9.1421
Epoch 005: Train MSE 2.0545 | Val MSE 4.6850 | Val R2 -1.4967
Epoch 010: Train MSE 1.8439 | Val MSE 16.8183 | Val R2 -7.9628
Epoch 015: Train MSE 1.6041 | Val MSE 2.9053 | Val R2 -0.5483
Epoch 020: Train MSE 1.4097 | Val MSE 1.4607 | Val R2 0.2216
Epoch 025: Train MSE 1.3130 | Val MSE 1.5957 | Val R2 0.1496
Epoch 030: Train MSE 1.2313 | Val MSE 1.6469 | Val R2 0.1223
Epoch 035: Train MSE 1.1566 | Val MSE 1.3112 | Val R2 0.3012
Epoch 040: Train MSE 1.1085 | Val MSE 1.7871 | Val R2 0.0476
Epoch 045: Train MSE 1.0454 | Val MSE 1.3635 | Val R2 0.2733
Epoch 050: Train MSE 0.9592 | Val MSE 1.2486 | Val R2 0.3346
Epoch 055: Train MSE 0.9124 | Val MSE 1.2222 | Val R2 0.3487
Epoch 059: Train MSE 0.8775 | Val MSE 1.2112 | Val R2 0.3545


2026/06/06 10:53:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/06 10:53:13 INFO mlflow.tracking._tracking_service.client: 🏃 View run Scaffold_Split_Clean_Desc at: http://localhost:5000/#/experiments/3/runs/99c9725d576747c7b8d2da67dd914975.
2026/06/06 10:53:13 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/3.



FINAL RESULT - MSE: 1.2112, MAE: 0.8035, R2: 0.3545
